# Этап 02. Полный каталог и рекомендованные данные БДМО

Просмотрены метаданные **всех 603 показателей**, а не прежние 22 кандидата. Для 29 содержательных кандидатов проверены фактические данные 2023–2024 и покрытие базы 2016 МО.

**Рекомендую 8 годовых признаков:** зарплата, интенсивность формальной занятости, плотность отчитывающихся организаций, доля прибыльных организаций, инвестиции на жителя, торговая площадь магазинов на жителя, плотность населения и доля освещённых улиц. Для них нужна очередь из 11 исходных показателей, уже заданная ниже.


## 2.1. Подготовка окружения и базовой панели

Загружаем конфигурацию БДМО, базовую панель муниципалитетов и таблицу сопоставления кодов. Инициализируем хранилище показателей и проверяем размер выборки.


In [1]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.bdmo_workflow import IndicatorStore, base_panel, load_mapping
from src.recommendations import read_plan, recommended_queue, search_catalog
config = json.loads((ROOT / 'configs/bdmo.json').read_text(encoding='utf-8'))
OUTPUT_DIR = ROOT / config['output_dir']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
panel, base_ids = base_panel(ROOT / config['base_panel'], config['expected_municipalities'], **config['calendar'])
mapping = load_mapping(ROOT / config['mapping_path'], base_ids, config['history_years'])
store = IndicatorStore(ROOT, config)
plan = read_plan(ROOT, config)
pd.set_option('display.max_colwidth', 120)
print(f'База: {len(base_ids)} МО x 24 месяца')


База: 2016 МО x 24 месяца


## 2.2. Каталог показателей и рекомендованная очередь

`source_hint_2023/2024` — исходные сведения справочника, **не покрытие наших 2016 МО**.
`base_coverage_2023/2024` — измеренное покрытие нашей базы для 29 проверенных показателей; для остальных оно неизвестно.
Каталог доступен офлайн. `refresh=True` обновляет метаданные источника; сохранённые эмпирические результаты относятся к версии от 28.09.2026 и требуют повторного аудита после обновления данных.

Загружаем каталог и формируем рекомендованную очередь; выводим состав показателей и доступные сведения об их покрытии.


In [2]:
catalog = store.catalog(refresh=False)
assert len(catalog) >= 603

queue = recommended_queue(catalog)

print("Полный каталог:", len(catalog))
print("В рекомендуемой очереди:", len(queue))
print("Доступные столбцы:", queue.columns.tolist())

cols = [
    "priority",
    "indicator_code",
    "feature_key",
    "indicator_name",
    "why",
    "base_coverage_2023",
    "base_coverage_2024",
]

display(queue[[c for c in cols if c in queue.columns]])

Полный каталог: 603
В рекомендуемой очереди: 11
Доступные столбцы: ['indicator_code', 'indicator_name', 'section', 'section_title', 'unit', 'frequency', 'available_years', 'dimensions', 'download_url', 'source_hint_2023', 'source_hint_2024', 'in_previous_shortlist', 'feature_key', 'view', 'match_score', 'priority', 'load_default', 'decision', 'why', 'role', 'transformation', 'caution', 'dimension_filters', 'period_preference', 'denominator']


,priority,indicator_code,feature_key,indicator_name,why
0,1,8112013,population_mean,Среднегодовая численность постоянного населения,Среднегодовое население — согласованный знаменатель годовых потоков.
1,2,8112027,population_jan,Оценка численности населения на 1 января текущего года,"Население на 1 января для масштаба, плотности и проверки демографических изменений."
2,3,8006001,land_area_ha,Общая площадь земель муниципального образования,Плотность расселения: площадь в гектарах конвертируется в км².
3,4,8423007,wage,Среднемесячная заработная плата работников организаций (без субъектов малого предпринимательства),Уровень оплаты труда дополняет структуру потребительских расходов.
4,5,8423005,employment,Среднесписочная численность работников организаций (без субъектов малого предпринимательства),Интенсивность формальной занятости на 1000 жителей.
5,6,8090001,reporting_companies,Количество хозяйствующих субъектов по данным бухгалтерской отчетности,Плотность отчитывающихся хозяйствующих субъектов; новый раздел 46.
6,7,8090008,profitable_share,Удельный вес прибыльных организаций в общем числе организаций по данным бухгалтерской отчетности,Доля прибыльных организаций: интенсивность без эффекта размера МО.
7,9,8109001,investment_total,"Инвестиции в основной капитал, осуществляемые организациями, находящимися на территории муниципального образования (...","Полные инвестиции организаций, а не только средства местного бюджета."
8,14,8002002,shop_floor,Площадь торгового зала объектов розничной торговли,Площадь торговых залов магазинов на жителя отражает физическое предложение торговли.
9,16,8006003,lit_street_km,"Общая протяженность освещенных частей улиц, проездов, набережных",Числитель доли освещённых улиц.


## 2.3. Импорт локальных исходных данных

В конфигурации `local_exports` указаны папки с исходными данными. Импорт читает CSV, Parquet, Excel и ZIP, сохраняет оригиналы, один раз разбивает строки по кодам и оставляет нужные годы. Повторный импорт использует контрольные суммы.
Если полный экспорт лежит в другом месте, добавьте его путь в список ниже. Старый canonical-файл без периодов и разрезов будет отмечен как непроверяемый: восстановить потерянные атрибуты нельзя. Он не удаляется и не выдаётся за исходные данные.

Импортируем доступные локальные исходные данные и выводим сводку по разделам и пропущенным файлам.


In [3]:
# Можно добавить путь к своему исходному полному экспорту:
# config['local_exports'].append('/путь/к/полному_экспорту.zip')
import_reports = store.import_existing()
print('Проверены источники:', config['local_exports'])
flat_reports = [item for report in import_reports for item in (report if isinstance(report, list) else [report])]
display(pd.DataFrame([{'source': r['source'], 'partitions': len(r['parts']), 'skipped_files': len(r['skipped'])} for r in flat_reports]))
# Подробности пропущенных непроверяемых файлов:
display([r['skipped'] for r in flat_reports if r['skipped']])


local_export_indexed 
Проверены источники: ['data/reviewed_raw', 'data/raw/bdmo', 'data/bdmo_export']


,source,partitions,skipped_files
0,/Users/olgakan/vlad/hack/data/reviewed_raw/7000004.parquet,1,0
1,/Users/olgakan/vlad/hack/data/reviewed_raw/7000005.parquet,1,0
2,/Users/olgakan/vlad/hack/data/reviewed_raw/8002001.parquet,1,0
3,/Users/olgakan/vlad/hack/data/reviewed_raw/8002002.parquet,1,0
4,/Users/olgakan/vlad/hack/data/reviewed_raw/8005005.parquet,1,0
5,/Users/olgakan/vlad/hack/data/reviewed_raw/8006001.parquet,1,0
6,/Users/olgakan/vlad/hack/data/reviewed_raw/8006003.parquet,1,0
7,/Users/olgakan/vlad/hack/data/reviewed_raw/8006007.parquet,1,0
8,/Users/olgakan/vlad/hack/data/reviewed_raw/8007013.parquet,1,0
9,/Users/olgakan/vlad/hack/data/reviewed_raw/8008019.parquet,1,0


[]

## 2.4. Загрузка одного показателя



In [4]:
result = store.load_next(queue, mapping=mapping)
if result is None:
    print('Рекомендуемая очередь уже обработана.')
else:
    meta, data_one, coverage_one = result
    print(meta['indicator_code'], meta['indicator_name'])
    display(coverage_one)


file_read_started Среднегодовая численность постоянного населения
file_read_complete Среднегодовая численность постоянного населения
complete Среднегодовая численность постоянного населения
8112013 Среднегодовая численность постоянного населения


,feature_key,view,requested_indicator,year,observed,universe,coverage,unmapped,outside_base
0,population_mean,demography,Среднегодовая численность постоянного населения,2023,1933,2016,0.958829,0,0
1,population_mean,demography,Среднегодовая численность постоянного населения,2024,1850,2016,0.917659,0,11


## 2.5. Последовательная загрузка рекомендованной очереди

Показатели обрабатываются по одному, каждый фиксируется в кэше до следующего. После ошибки повторный запуск продолжает с незавершённого; готовые результаты не скачиваются заново.


In [5]:
RUN_ALL_RECOMMENDED = True
if RUN_ALL_RECOMMENDED:
    while True:
        result = store.load_next(queue, mapping=mapping)
        if result is None: break
        meta, _, coverage_one = result
        print('Готов:', meta['indicator_code'], meta['feature_key'])
        display(coverage_one)


file_read_started Оценка численности населения на 1 января текущего года
file_read_complete Оценка численности населения на 1 января текущего года
complete Оценка численности населения на 1 января текущего года
Готов: 8112027 population_jan


,feature_key,view,requested_indicator,year,observed,universe,coverage,unmapped,outside_base
0,population_jan,demography,Оценка численности населения на 1 января текущего года,2023,2011,2016,0.997520,0,0
1,population_jan,demography,Оценка численности населения на 1 января текущего года,2024,2015,2016,0.999504,0,0


file_read_started Общая площадь земель муниципального образования
file_read_complete Общая площадь земель муниципального образования
complete Общая площадь земель муниципального образования
Готов: 8006001 land_area_ha


,feature_key,view,requested_indicator,year,observed,universe,coverage,unmapped,outside_base
0,land_area_ha,demography,Общая площадь земель муниципального образования,2023,2016,2016,1.000000,0,0
1,land_area_ha,demography,Общая площадь земель муниципального образования,2024,1997,2016,0.990575,0,0


file_read_started Среднемесячная заработная плата работников организаций (без субъектов малого предпринимательства)
file_read_complete Среднемесячная заработная плата работников организаций (без субъектов малого предпринимательства)
complete Среднемесячная заработная плата работников организаций (без субъектов малого предпринимательства)
Готов: 8423007 wage


...

,feature_key,view,requested_indicator,year,observed,universe,coverage,unmapped,outside_base
0,shop_floor,retail_supply,Площадь торгового зала объектов розничной торговли,2023,1882,2016,0.933532,0,0
1,shop_floor,retail_supply,Площадь торгового зала объектов розничной торговли,2024,1887,2016,0.936012,0,0


file_read_started Общая протяженность освещенных частей улиц, проездов, набережных
file_read_complete Общая протяженность освещенных частей улиц, проездов, набережных
complete Общая протяженность освещенных частей улиц, проездов, набережных
Готов: 8006003 lit_street_km


,feature_key,view,requested_indicator,year,observed,universe,coverage,unmapped,outside_base
0,lit_street_km,infrastructure,"Общая протяженность освещенных частей улиц, проездов, набережных",2023,1914,2016,0.949405,0,0
1,lit_street_km,infrastructure,"Общая протяженность освещенных частей улиц, проездов, набережных",2024,1895,2016,0.939980,0,0


file_read_started Общая протяженность улиц, проездов, набережных
file_read_complete Общая протяженность улиц, проездов, набережных
complete Общая протяженность улиц, проездов, набережных
Готов: 8006007 street_km


,feature_key,view,requested_indicator,year,observed,universe,coverage,unmapped,outside_base
0,street_km,infrastructure,"Общая протяженность улиц, проездов, набережных",2023,1916,2016,0.950397,0,0
1,street_km,infrastructure,"Общая протяженность улиц, проездов, набережных",2024,1898,2016,0.941468,0,0


## 2.6. Экспорт завершённых показателей

Экспортируются только завершённые результаты. Счётчик явно показывает, вся ли очередь готова. Новые результаты идут в `outputs/bdmo_v3`, прежние экспорты сохраняются.


In [6]:
ready = store.completed_selection(queue)
print(f'Завершено {len(ready)} из {len(queue)}')
if len(ready):
    data = store.export(ready, OUTPUT_DIR)
    print('Строк:', len(data), 'Папка:', OUTPUT_DIR)
else:
    print('Пока нет завершённых показателей.')


Завершено 11 из 11
Строк: 43308 Папка: /Users/olgakan/vlad/hack/outputs/bdmo_v3


## 2.7. Анализ дополнительных показателей


In [8]:
optional = catalog[
    catalog['role'].isin([
        'candidate',
        'alternative',
        'diagnostic',
        'support_optional'
    ])
]

optional_cols = [
    'indicator_code',
    'indicator_name',
    'why',
    'caution',
    'base_coverage_2023',
    'base_coverage_2024'
]

display(optional[
    [c for c in optional_cols if c in optional.columns]
])

education = search_catalog(catalog, text='образован')

education_cols = [
    'indicator_code',
    'indicator_name',
    'frequency',
    'decision',
    'why'
]

display(education[
    [c for c in education_cols if c in education.columns]
])

,indicator_code,indicator_name,why,caution
7,7000005,"Выручка (нетто) от продажи товаров, продукции, работ, услуг (за минусом налога на добавленную стоимость, акцизов и и...",Масштаб деловой активности на жителя из бухгалтерской отчётности.,Расположение головных организаций и крупные выбросы; вероятна корреляция с числом фирм.
9,8109003,Объем инвестиций в основной капитал (за исключением бюджетных средств) в расчете на одного человека,Прямая интенсивность внебюджетных инвестиций; альтернатива общим инвестициям.,Другой состав инвестиций; не считать дубликатом общего итога.
10,8019014,Объем социальных выплат населению и налогооблагаемых денежных доходов населения,Социальные выплаты и налогооблагаемые денежные доходы на жителя расширяют контекст благосостояния.,Не полные располагаемые доходы и не медианный доход.
11,8211001,"Общая площадь жилых помещений, приходящаяся в среднем на одного жителя — всего",Обеспеченность жильём на жителя — отдельная характеристика среды.,Большая площадь не обязательно означает высокое качество жилья.
12,8215001,"Общая площадь жилых помещений, введенная в действие за год, приходящаяся в среднем на одного жителя",Жилищное строительство на жителя как динамика развития.,Нули сохраняются; годовые всплески и неполное покрытие.
14,8801004,"Общий объем всех продовольственных товаров, реализованных в границах муниципального образования",Реализация продовольственных товаров на жителя — внешняя проверка Сбер-структуры.,Может дублировать потребительский view; сначала использовать для интерпретации кластеров.
17,8106005,"Доля протяженности автомобильных дорог общего пользования местного значения, не отвечающих нормативным требованиям, ...","Качество дорожной инфраструктуры, отдельное от транспортных расстояний Сбера.",Различия отчётности и неполное покрытие.
18,8313015,Доля налоговых и неналоговых доходов местного бюджета (за исключением поступлений налоговых доходов по дополнительны...,Действующий показатель фискальной самостоятельности взамен устаревшего раздела 13.,"Доля в собственных доходах без субвенций, не в общем доходе бюджета."
19,8060001,Число коллективных средств размещения,Плотность коллективных средств размещения для туристической специализации.,Пропуск не означает отсутствие туризма; отдельный дополнительный view.
20,7000004,Число ночевок в коллективных средствах размещения,"Туристическая интенсивность по ночёвкам, новая содержательная проверка сезонности.",Слабое покрытие может делать общий 2016-объектный view непригодным.


,indicator_code,indicator_name,frequency,decision,why
2,8006001,Общая площадь земель муниципального образования,Годовая,support,Плотность расселения: площадь в гектарах конвертируется в км².
8,8109001,"Инвестиции в основной капитал, осуществляемые организациями, находящимися на территории муниципального образования (...","Годовая, январь-декабрь",primary,"Полные инвестиции организаций, а не только средства местного бюджета."
14,8801004,"Общий объем всех продовольственных товаров, реализованных в границах муниципального образования",Годовая,candidate,Реализация продовольственных товаров на жителя — внешняя проверка Сбер-структуры.
18,8313015,Доля налоговых и неналоговых доходов местного бюджета (за исключением поступлений налоговых доходов по дополнительны...,Годовая,candidate,Действующий показатель фискальной самостоятельности взамен устаревшего раздела 13.
68,8006006,Площадь земель сельхозугодий муниципального образования,Годовая,no_focus_data,В справочнике нет данных за 2023 и 2024; не приоритет для выбранного периода.
69,8006008,"Протяженность автодорог общего пользования местного значения, находящихся в собственности муниципальных образований ...",Годовая,no_focus_data,В справочнике нет данных за 2023 и 2024; не приоритет для выбранного периода.
124,8109011,"Инвестиции в основной капитал, осуществляемые организациями, находящимися на территории муниципального образования (...","Годовая, январь-декабрь",no_focus_data,В справочнике нет данных за 2023 и 2024; не приоритет для выбранного периода.
127,8109015,"Средства на долевое строительство населения и организаций, находящихся на территории муниципального образования — из...","Годовая, январь-декабрь",no_focus_data,В справочнике нет данных за 2023 и 2024; не приоритет для выбранного периода.
136,8013003,"Профицит, дефицит (-) бюджета муниципального образования (местного бюджета), фактически исполнено",Годовая,no_focus_data,В справочнике нет данных за 2023 и 2024; не приоритет для выбранного периода.
137,8013004,Расходы бюджета муниципального образования на содержание работников органов местного самоуправления в расчете на одн...,Годовая,no_focus_data,В справочнике нет данных за 2023 и 2024; не приоритет для выбранного периода.
